In [ ]:
import matplotlib.pyplot as plt
import pandas
import functools 
import datetime

import pmana.utils

In [ ]:
TIME_DIR = "20260202"
TIME_RANGE = "20260202_20260210"

#### Parse Padova test-stand data

In [ ]:
PATH_INPUT  = f"/Users/triozzi/Downloads/20260202_202602xx_TestPulse"
PATH_TARGET = f"../data/padova/{TIME_DIR}/DataPadova_{TIME_RANGE}"

pmana.utils.io.FormatPadovaData(
    PATH_INPUT,  ###< raw Padova test-stand campaign
    PATH_TARGET  ###< target path for the restructured data
)

#### Look at one single measurement

In [ ]:
fig, ax = plt.subplots(figsize=(6,3))

MEASUREMENT = '00464'
# extract a measurement 
Data = pmana.utils.io.ExtractSingleMeasurement(
    f"../data/padova/{TIME_DIR}/DataPadova_{TIME_RANGE}/{MEASUREMENT}"
)
print(f"There were {len(Data)} used channels in this measurement.")

CHs = [1, 2, 3]
for i, chname in enumerate(CHs):
    # plot channel data
    pmana.utils.plotting.PlotSingleChannel(
        Data[i],
        ax,
        channel = i,
        rebin = False
    )

ax = pmana.utils.plotting.UpdateMatplotlibStyle(
    ax,
    "Integral [V*t/R] / Pulse height [V]",
    "Counts [#]"
)

ax.legend()
ax.set_title(f"{TIME_RANGE}/{MEASUREMENT}")
# ax.set_xlim(0.18, 0.1875)
# ax.set_xlim(0.0312, 0.0317)
plt.show()

#### Analyze a campaign

In [ ]:
PATH_CAMPAIGN = f"../data/padova/{TIME_DIR}/DataPadova_{TIME_RANGE}"
PATH_TEMPERATURES = f"../data/padova/{TIME_DIR}/PM_Temperature_{TIME_RANGE}.txt"
PATH_TIMES = f"../data/padova/{TIME_DIR}/PadovaData_{TIME_RANGE}_TimeMapping.txt"

In [ ]:
# get temperature mapping
Temperatures = pmana.utils.io.ExtractTemperatureMonitoring(
    PATH_TEMPERATURES
)

# get time mapping
TimeMapping = pmana.utils.io.ExtractFileTimes(
    PATH_TIMES
)

In [ ]:
# analyze campaign
Output = pmana.utils.anatestdata.Iterate(
    PATH_CAMPAIGN,                                                                ###< path to restructured data
    # pmana.utils.anatestdata.AnalyzeMeasurement,                                 ###< analyzing module 
    functools.partial(pmana.utils.anatestdata.GaussianFitToChannel, rebin=False), ###< analyzing module, if you want to change some options 
    TimeMapping                                                                   ###< file-to-time mapping
)

# convert top DataFrame
Output = pandas.DataFrame(Output)

# re-format the dataframe
Output.columns = ["Peak_CH1", "Err_Peak_CH1", "Width_CH1", "Err_Width_CH1",
                  "Peak_CH2", "Err_Peak_CH2", "Width_CH2", "Err_Width_CH2", 
                  "Peak_CH3", "Err_Peak_CH3", "Width_CH3", "Err_Width_CH3", 
                #   "Peak_CH4", "Err_Peak_CH4", "Width_CH4", "Err_Width_CH4", 
                #   "Peak_CH5", "Err_Peak_CH5", "Width_CH5", "Err_Width_CH5", 
                #   "Peak_CH6", "Err_Peak_CH6", "Width_CH6", "Err_Width_CH6", 
                #   "Peak_CH7", "Err_Peak_CH7", "Width_CH7", "Err_Width_CH7",
                #   "Peak_CH8", "Err_Peak_CH8", "Width_CH8", "Err_Width_CH8",  
                  "Date", "Number"]

# sort by date
Output.sort_values(
    by = 'Date', 
    inplace = True,
    ignore_index = True
)

In [ ]:
# temporarily drop NaNs... ideally, actually resolve the issue
# by retaining time even when not all channels are available
Output = Output.dropna(
  how = 'any',
  subset = ['Date']
)

# temporarily drop zero values from bad fits... ideally,
# you should fix the failing fits rather than skipping them
# Output = Output[Output.Peak_CH3 > 0]

# properly handle data types
# not always needed
# Vars = Output.columns[:-2]
# Output[Vars] = Output[Vars].astype(float)

In [ ]:
Output = Output[Output.Date > datetime.datetime(2025, 11, 24, 12)]

In [ ]:
fig, ax = plt.subplots(figsize=(6, 2.*3), nrows=3, layout='tight')

# CH1
ax[0].plot(Output.Date, Output.Peak_CH1, lw=1)
ax[0].fill_between(Output.Date, Output.Peak_CH1-Output.Err_Peak_CH1,  Output.Peak_CH1+Output.Err_Peak_CH1, alpha=0.3, label='F1 $\\pm1\\sigma$')
pmana.utils.plotting.UpdateMatplotlibStyle(ax[0], None, 'Peak [V]')
ax[0].set_xticklabels([])

# CH2
ax[1].plot(Output.Date, Output.Peak_CH2, lw=1)
ax[1].fill_between(Output.Date, Output.Peak_CH2-Output.Err_Peak_CH2,  Output.Peak_CH2+Output.Err_Peak_CH2, alpha=0.3, label='F3 $\\pm1\\sigma$')
pmana.utils.plotting.UpdateMatplotlibStyle(ax[1], None, 'Peak [V]')
ax[1].set_ylim(0.1855, 0.1875)
ax[1].set_xticklabels([])

# CH3
ax[2].plot(Output.Date, Output.Peak_CH3, lw=1)
ax[2].fill_between(Output.Date, Output.Peak_CH3-Output.Err_Peak_CH3,  Output.Peak_CH3+Output.Err_Peak_CH3, alpha=0.3, label='F4 $\\pm1\\sigma$')
pmana.utils.plotting.UpdateMatplotlibStyle(ax[2], None, 'Peak [V]')
# ax[0].set_xticklabels([])

for a in ax:
    # gfx
    a.legend(frameon=True, fancybox=False, handlelength=1, fontsize=8, loc='upper left')
    a.tick_params(axis='x', labelrotation=30)
    a.set_xlim(Output['Date'].iloc[0], Output['Date'].iloc[-1])

    # plot temperatures
    a2 = a.twinx()
    a2.fill_between(Temperatures.Date, y1=Temperatures.T1, y2=Temperatures.T2, fc='red', alpha=0.4, label='Temperature', zorder=-3)
    pmana.utils.plotting.UpdateMatplotlibStyle(a2, None, 'T [°C]')
    a2.legend(frameon=True, fancybox=False, handlelength=1, fontsize=8, loc='lower left')

    # a.set_xlim(datetime.datetime(2026, 1, 19, 11), datetime.datetime(2026, 1, 21, 20))

plt.show()
fig.savefig(f"../plots/{TIME_RANGE}.png", dpi=300, bbox_inches='tight')

##### Multiple-column plot example

In [ ]:
fig, ax = plt.subplots(figsize=(6*2, 2.*4), ncols=2, nrows=4, layout='tight')

# CH1
ax[0, 0].plot(Output.Date, Output.Peak_CH1, lw=1)
ax[0, 0].fill_between(Output.Date, Output.Peak_CH1-Output.Err_Peak_CH1,  Output.Peak_CH1+Output.Err_Peak_CH1, alpha=0.3, label='CH1 $\\pm1\\sigma$')
pmana.utils.plotting.UpdateMatplotlibStyle(ax[0, 0], None, 'Integral [V*t/R]')
ax[0, 0].set_xticklabels([])

# CH2
ax[1, 0].plot(Output.Date, Output.Peak_CH2, lw=1)
ax[1, 0].fill_between(Output.Date, Output.Peak_CH2-Output.Err_Peak_CH2,  Output.Peak_CH2+Output.Err_Peak_CH2, alpha=0.3, label='CH2 $\\pm1\\sigma$')
pmana.utils.plotting.UpdateMatplotlibStyle(ax[1, 0], None, 'Integral [V*t/R]')
ax[1, 0].set_xticklabels([])

# CH3
ax[2, 0].plot(Output.Date, Output.Peak_CH3.astype(float), lw=1)
ax[2, 0].fill_between(Output.Date, Output.Peak_CH3-Output.Err_Peak_CH3,  Output.Peak_CH3+Output.Err_Peak_CH3, alpha=0.3, label='CH3 $\\pm1\\sigma$')
pmana.utils.plotting.UpdateMatplotlibStyle(ax[2, 0], None, 'Peak [V]')
ax[2, 0].set_xticklabels([])

# CH4
ax[3, 0].plot(Output.Date, Output.Peak_CH4, lw=1)
ax[3, 0].fill_between(Output.Date, Output.Peak_CH4-Output.Err_Peak_CH4,  Output.Peak_CH4+Output.Err_Peak_CH4, alpha=0.3, label='CH4 $\\pm1\\sigma$')
pmana.utils.plotting.UpdateMatplotlibStyle(ax[3, 0], None, 'Peak [V]')

# CH5
ax[0, 1].plot(Output.Date, Output.Peak_CH5, lw=1)
ax[0, 1].fill_between(Output.Date, Output.Peak_CH5-Output.Err_Peak_CH5,  Output.Peak_CH5+Output.Err_Peak_CH5, alpha=0.3, label='CH3 $\\pm1\\sigma$')
pmana.utils.plotting.UpdateMatplotlibStyle(ax[0, 1], None, 'Peak-to-peak [V]')
ax[0, 1].set_xticklabels([])

# CH6
ax[1, 1].plot(Output.Date, Output.Peak_CH6, lw=1)
ax[1, 1].fill_between(Output.Date, Output.Peak_CH6-Output.Err_Peak_CH6,  Output.Peak_CH6+Output.Err_Peak_CH6, alpha=0.3, label='CH4 $\\pm1\\sigma$')
pmana.utils.plotting.UpdateMatplotlibStyle(ax[1, 1], None, 'Peak-to-peak [V]')
ax[1, 1].set_xticklabels([])

# CH5
ax[2, 1].plot(Output.Date, Output.Peak_CH7, lw=1)
ax[2, 1].fill_between(Output.Date, Output.Peak_CH7-Output.Err_Peak_CH7,  Output.Peak_CH7+Output.Err_Peak_CH7, alpha=0.3, label='CH1 $\\pm1\\sigma$')
pmana.utils.plotting.UpdateMatplotlibStyle(ax[2, 1], None, 'Peak-to-peak [V]')
ax[2, 1].set_xticklabels([])

# CH5
ax[3, 1].plot(Output.Date, Output.Peak_CH8, lw=1)
ax[3, 1].fill_between(Output.Date, Output.Peak_CH8-Output.Err_Peak_CH8,  Output.Peak_CH8+Output.Err_Peak_CH8, alpha=0.3, label='CH2 $\\pm1\\sigma$')
pmana.utils.plotting.UpdateMatplotlibStyle(ax[3, 1], None, 'Peak-to-peak [V]')

for row in ax:
    for a in row:
        # gfx
        a.legend(frameon=True, fancybox=False, handlelength=1, fontsize=8, loc='upper left')
        a.tick_params(axis='x', labelrotation=30)
        a.set_xlim(Output['Date'].iloc[0], Output['Date'].iloc[-1])

        # plot temperatures
        a2 = a.twinx()
        a2.fill_between(Temperatures.Date, y1=Temperatures.T1, y2=Temperatures.T2, fc='red', alpha=0.4, label='Temperature', zorder=-3)
        pmana.utils.plotting.UpdateMatplotlibStyle(a2, None, 'T [°C]')
        a2.legend(frameon=True, fancybox=False, handlelength=1, fontsize=8, loc='lower left')

plt.show()
fig.savefig(f"../plots/{TIME_RANGE}.png", dpi=300, bbox_inches='tight')

##### Dump analysis outcome to external files

In [ ]:
# dump single channels to file
Output[['Date', 'Peak_CH1']].to_csv(
   f'../out/{TIME_RANGE}_Complete_CH1.txt',
   index = False,
   sep = ' ',
   header = ['Date', 'Peak']
)
Output[['Date', 'Peak_CH2']].to_csv(
   f'../out/{TIME_RANGE}_Complete_CH2.txt',
   index = False,
   sep = ' ',
   header = ['Date', 'Peak']
)
Output[['Date', 'Peak_CH3']].to_csv(
   f'../out/{TIME_RANGE}_Complete_CH3.txt',
   index = False,
   sep = ' ',
   header = ['Date', 'Peak']
)

In [ ]:
# extract gain
# K = 20.6
# Output['Gain'] = K * (Output['Peak_CH3'] / Output['Peak_CH4'])
# Output['Err_Gain'] = Output['Gain'] * Output['Peak_CH4']

# dump it to file
VarsToDump = ['Peak_CH1', 'Peak_CH2', 'Peak_CH3']
Output[VarsToDump].to_csv(
   f'../out/{TIME_RANGE}_Gain.txt',
   index = False,
   sep = ',',
   header = ['CH1', 'CH2', 'CH3']
)

# dump single channels to file
Output[['Peak_CH1']].to_csv(
   f'../out/{TIME_RANGE}_CH1.txt',
   index = False,
   decimal = ',',
   sep = ' ',
   header = ['CH1']
)
Output[['Peak_CH2']].to_csv(
   f'../out/{TIME_RANGE}_CH2.txt',
   index = False,
   decimal = ',',
   sep = ' ',
   header = ['CH2']
)
Output[['Peak_CH3']].to_csv(
   f'../out/{TIME_RANGE}_CH3.txt',
   index = False,
   decimal = ',',
   sep = ' ',
   header = ['CH3']
)
# Output[['Peak_CH4']].to_csv(
#    f'../out/{TIME_RANGE}_CH4.txt',
#    index = False,
#    decimal = ',',
#    sep = ' ',
#    header = ['CH4']
# )
# Output[['Peak_CH5']].to_csv(
#    f'../out/{TIME_RANGE}_CH5.txt',
#    index = False,
#    decimal = ',',
#    sep = ' ',
#    header = ['CH5']
# )
# Output[['Peak_CH6']].to_csv(
#    f'../out/{TIME_RANGE}_CH6.txt',
#    index = False,
#    decimal = ',',
#    sep = ' ',
#    header = ['CH6']
# )
# Output[['Peak_CH7']].to_csv(
#    f'../out/{TIME_RANGE}_CH7.txt',
#    index = False,
#    decimal = ',',
#    sep = ' ',
#    header = ['CH7']
# )
# Output[['Peak_CH8']].to_csv(
#    f'../out/{TIME_RANGE}_CH8.txt',
#    index = False,
#    decimal = ',',
#    sep = ' ',
#    header = ['CH8']
# )

Output.head()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 2.5), nrows=1, layout='tight')

# CH1
ax.plot(Output.Date, Output.Gain, lw=1, label='20.6$\\times$(CH3/CH8)')
# ax.fill_between(Output.Date, Output.Gain-Output.Err_Gain,  Output.Gain+Output.Err_Gain, alpha=0.3, label='20.6$\\times$(CH3/CH8) $\\pm1\\sigma$')
pmana.utils.plotting.UpdateMatplotlibStyle(ax, None, 'Gain')

# gfx
ax.legend(frameon=True, fancybox=False, handlelength=1, fontsize=8, loc='upper left')
ax.tick_params(axis='x', labelrotation=30)
ax.set_xlim(Output['Date'].iloc[0], Output['Date'].iloc[-1])

# plot temperatures
a2 = ax.twinx()
a2.fill_between(Temperatures.Date, y1=Temperatures.T1, y2=Temperatures.T2, fc='red', alpha=0.4, label='Temperature', zorder=-3)
pmana.utils.plotting.UpdateMatplotlibStyle(a2, None, 'T [°C]')
a2.legend(frameon=True, fancybox=False, handlelength=1, fontsize=8, loc='lower left')

plt.show()
fig.savefig(f"../plots/{TIME_RANGE}_Gain.png", dpi=300, bbox_inches='tight')

#### Merge all campaigns

This needs more work, to account for the fact that channels have different meanings in different measurement campaigns.

In [ ]:
PATH_DATA = "../data"

In [ ]:
# analyze all campaigns together
Output, Temperatures = pmana.utils.anatestdata.MergeCampaigns(
    PATH_DATA,                                      ###< top path to restructured data
    pmana.utils.io.ExtractFileTimes,                ###< analyzer to handle time mapping
    pmana.utils.io.ExtractTemperatureMonitoring,    ###< analyzer to handle temperatures
    pmana.utils.anatestdata.Iterate,                ###< main loop to each campaign
    pmana.utils.anatestdata.GaussianFitToChannel    ###< analyzer to each measurement
)

# format the output nicely
Output = pandas.DataFrame(Output)
Output.columns = ["Peak_CH1", "Err_Peak_CH1", "Width_CH1", "Err_Width_CH1",
                  "Peak_CH2", "Err_Peak_CH2", "Width_CH2", "Err_Width_CH2", 
                  "Peak_CH3", "Err_Peak_CH3", "Width_CH3", "Err_Width_CH3", 
                  "Peak_CH4", "Err_Peak_CH4", "Width_CH4", "Err_Width_CH4", 
                  "Peak_CH5", "Err_Peak_CH5", "Width_CH5", "Err_Width_CH5", 
                  "Peak_CH6", "Err_Peak_CH6", "Width_CH6", "Err_Width_CH6", 
                  "Peak_CH7", "Err_Peak_CH7", "Width_CH7", "Err_Width_CH7", 
                  "Peak_CH8", "Err_Peak_CH8", "Width_CH8", "Err_Width_CH8", 
                  "Date", "Number"]
Output.sort_values(
    by = 'Date', 
    inplace = True,
    ignore_index = True
)

# format the temperatures nicely
Temperatures.sort_values(
    by = 'Date', 
    inplace = True,
    ignore_index = True
)

In [ ]:
Output

In [ ]:
fig, ax = plt.subplots(figsize=(6, 2.5*2), nrows=2, layout='tight')

# CH1
ax[0].plot(Output.Date, Output.Peak_CH1, lw=1)
ax[0].fill_between(Output.Date, Output.Peak_CH1-Output.Err_Peak_CH1,  Output.Peak_CH1+Output.Err_Peak_CH1, alpha=0.3, label='CH1 $\\pm1\\sigma$')
pmana.utils.plotting.UpdateMatplotlibStyle(ax[0], None, 'Integral [V*t/R]')

# CH2
ax[1].plot(Output.Date, Output.Peak_CH2, lw=1)
ax[1].fill_between(Output.Date, Output.Peak_CH2-Output.Err_Peak_CH2,  Output.Peak_CH2+Output.Err_Peak_CH2, alpha=0.3, label='CH3 $\\pm1\\sigma$')
pmana.utils.plotting.UpdateMatplotlibStyle(ax[1], None, 'Peak [V]')

for a in ax:
    # gfx
    a.legend(frameon=True, fancybox=False, handlelength=1, fontsize=8, loc='upper left')
    a.tick_params(axis='x', labelrotation=30)
    # a.set_xlim(Output['Date'].iloc[0], Output['Date'].iloc[-1])

    # plot temperatures
    a2 = a.twinx()
    a2.fill_between(Temperatures.Date, y1=Temperatures.T1, y2=Temperatures.T2, fc='red', alpha=0.4, label='Temperature', zorder=-3)
    pmana.utils.plotting.UpdateMatplotlibStyle(a2, None, 'T [°C]')
    a2.legend(frameon=True, fancybox=False, handlelength=1, fontsize=8, loc='lower left')

plt.show()